# KuaiRand Two-Tower Gold Data Preparation

This notebook validates and explains the Gold dataset produced by `scripts/build_kuairand_two_tower_gold.py`. Heavy transformation logic lives in `recommender/gold/two_tower.py` so the pipeline is reproducible outside notebooks.

The deterministic feature-analysis sample is not used here. The source of truth is the full Silver event stream.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'recommender').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from recommender.spark import get_spark

SILVER_DIR = PROJECT_ROOT / 'data/silver/kuairand'
GOLD_DIR = PROJECT_ROOT / 'data/gold/two_tower/v1'

spark = get_spark('kuairand-two-tower-gold-validation', reset=True)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} | silver={SILVER_DIR} | gold={GOLD_DIR}')

## Build command

Run this from the repository root after Silver exists. On a local machine with tight disk, reduce Spark concurrency and make sure there is enough free space for Spark spill and Parquet output.

In [ ]:
print('''SPARK_MASTER=local[2] PYSPARK_SUBMIT_ARGS="--driver-memory 10g pyspark-shell" \\
python scripts/build_kuairand_two_tower_gold.py \\
  --silver-dir data/silver/kuairand \\
  --output-dir data/gold/two_tower/v1 \\
  --overwrite''')

## Manifest and split summary

In [ ]:
manifest_path = GOLD_DIR / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing manifest. Build Gold first: {manifest_path}')

manifest = json.loads(manifest_path.read_text())
print(json.dumps({
    'version': manifest['gold_dataset_version'],
    'temporal_split': manifest['temporal_split'],
    'example_summary': manifest['example_summary'],
    'cold_start': manifest['cold_start'],
}, indent=2))

## Gold schemas

In [ ]:
tables = {
    'train_core': GOLD_DIR / 'train',
    'validation_core': GOLD_DIR / 'validation',
    'test_core': GOLD_DIR / 'test',
    'train_user_state': GOLD_DIR / 'user_state/train',
    'train_item_point_in_time': GOLD_DIR / 'item_features/point_in_time/train',
    'item_static': GOLD_DIR / 'item_features/static',
    'train_targets': GOLD_DIR / 'targets/train',
    'history_events': GOLD_DIR / 'history/events',
}

for name, path in tables.items():
    df = spark.read.parquet(str(path))
    print(f'\n=== {name}: {df.count():,} rows ===')
    df.printSchema()

## Target distribution

In [ ]:
for split in ['train', 'validation', 'test']:
    target = spark.read.parquet(str(GOLD_DIR / 'targets' / split))
    print(f'\n{split}')
    target.groupBy('target_class').agg(
        F.count('*').alias('rows'),
        F.avg('engagement_strength').alias('avg_engagement_strength'),
    ).orderBy('target_class').show(truncate=False)

## Feature catalog

In [ ]:
catalog = json.loads((GOLD_DIR / 'feature_catalog.json').read_text())['features']
catalog_df = spark.createDataFrame(catalog)
catalog_df.groupBy('tower_assignment').count().orderBy('tower_assignment').show(truncate=False)
catalog_df.select(
    'feature_name', 'tower_assignment', 'static_vs_dynamic', 'point_in_time_availability', 'leakage_risk', 'reason'
).orderBy('tower_assignment', 'feature_name').show(120, truncate=False)

## History examples and leakage checks

In [ ]:
train = spark.read.parquet(str(GOLD_DIR / 'train'))
history = spark.read.parquet(str(GOLD_DIR / 'history/events'))

train.select(
    'example_id', 'context_id', 'user_id', 'video_id', 'session_id',
    'user_event_index', 'history_end_user_event_index',
    'session_event_index', 'history_end_session_event_index', 'as_of_time'
).orderBy('user_id', 'user_event_index').show(10, truncate=False)

checks = {
    'duplicate_example_ids': train.groupBy('example_id').count().where(F.col('count') > 1).count(),
    'bad_user_history_refs': train.where(F.col('history_end_user_event_index') >= F.col('user_event_index')).count(),
    'bad_session_history_refs': train.where(F.col('history_end_session_event_index') >= F.col('session_event_index')).count(),
}
checks

## Vocabularies and numerical transforms

In [ ]:
print(json.dumps(json.loads((GOLD_DIR / 'vocabularies/manifest.json').read_text()), indent=2)[:4000])
print(json.dumps(json.loads((GOLD_DIR / 'transforms/numeric_stats.json').read_text()), indent=2)[:4000])